# mini-infer vs HuggingFace Benchmark on Colab

这个 notebook 用于在 Google Colab GPU 环境下对比：

- 当前框架 `mini-infer` 的 `LLMEngine.generate()`
- HuggingFace `AutoModelForCausalLM.generate()` baseline

默认使用 `Qwen/Qwen2.5-0.5B-Instruct`，并运行项目里的：

- `benchmarks/benchmark_mini.py`
- `benchmarks/benchmark_hf.py`


## 使用说明

1. 在 Colab 中打开本 notebook
2. 选择 `Runtime -> Change runtime type -> GPU`
3. 依次运行全部单元

如果 `flash-attn` 安装失败，`mini-infer` 的 CUDA benchmark 可能无法运行；这种情况下可以先只跑 HuggingFace baseline。

In [ ]:
!nvidia-smi || true

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/shallowdream2/mini-infer.git"
REPO_DIR = Path("/content/mini-infer")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

%cd /content/mini-infer
!git rev-parse --short HEAD

In [ ]:
!python -m pip install -U pip setuptools wheel
!python -m pip install -e .
!python -m pip install pandas tabulate

In [ ]:
import subprocess
import sys

flash_attn_ok = False
try:
    import flash_attn  # noqa: F401
    flash_attn_ok = True
    print("flash-attn 已安装")
except Exception:
    print("开始安装 flash-attn ...")
    cmd = [sys.executable, "-m", "pip", "install", "flash-attn", "--no-build-isolation"]
    result = subprocess.run(cmd, check=False)
    flash_attn_ok = result.returncode == 0
    print(f"flash-attn 安装成功: {flash_attn_ok}")

In [ ]:
import torch
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

In [ ]:
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE = "cuda:0"
DTYPE = "float16"
BATCH_SIZE = 4
MAX_NEW_TOKENS = 64
NUM_GPU_BLOCKS = 256

print({
    "MODEL": MODEL,
    "DEVICE": DEVICE,
    "DTYPE": DTYPE,
    "BATCH_SIZE": BATCH_SIZE,
    "MAX_NEW_TOKENS": MAX_NEW_TOKENS,
    "NUM_GPU_BLOCKS": NUM_GPU_BLOCKS,
})

In [ ]:
from benchmarks.benchmark_mini import benchmark_mini

if not flash_attn_ok:
    raise RuntimeError("flash-attn 未安装成功，无法运行 mini-infer CUDA benchmark。")

mini_result = benchmark_mini(
    model_name=MODEL,
    batch_size=BATCH_SIZE,
    max_new_tokens=MAX_NEW_TOKENS,
    device=DEVICE,
    dtype=DTYPE,
    num_gpu_blocks=NUM_GPU_BLOCKS,
)
mini_result

In [ ]:
from benchmarks.benchmark_hf import benchmark_hf

hf_result = benchmark_hf(
    model_name=MODEL,
    batch_size=BATCH_SIZE,
    max_new_tokens=MAX_NEW_TOKENS,
    device=DEVICE,
    dtype=DTYPE,
)
hf_result

In [ ]:
import pandas as pd

rows = [
    {
        "engine": "mini-infer",
        "ttft_ms": round(mini_result.ttft_ms, 2),
        "tpot_ms": round(mini_result.tpot_ms, 2),
        "throughput_tok_s": round(mini_result.throughput_tok_s, 2),
        "peak_mem_gb": round(mini_result.peak_memory_gb, 2),
    },
    {
        "engine": "huggingface",
        "ttft_ms": round(hf_result.ttft_ms, 2),
        "tpot_ms": round(hf_result.tpot_ms, 2),
        "throughput_tok_s": round(hf_result.throughput_tok_s, 2),
        "peak_mem_gb": round(hf_result.peak_memory_gb, 2),
    },
]

df = pd.DataFrame(rows)
display(df)

mini_vs_hf = mini_result.throughput_tok_s / hf_result.throughput_tok_s if hf_result.throughput_tok_s else 0.0
print(f"mini-infer throughput / HF throughput = {mini_vs_hf:.3f}x")

## 备注

- `benchmark_mini.py` 在 CUDA 路径下要求 `flash-attn`
- Colab 免费环境的 GPU 型号不固定，结果会有波动
- 如果 0.5B 运行稳定，可以继续把 `MODEL` 改成 `Qwen/Qwen2.5-1.5B-Instruct`
- 如果显存不足，请优先减小 `BATCH_SIZE` 和 `MAX_NEW_TOKENS`
